# Reddit Logs to YTMusic Playlists

In [1]:
# Set Paths

# Path to header .json file for ytmusic api
ytmusic_header_path = '..'
# Path to .tsv reddit search logs
reddit_log_path = '..\\..\\reddit_scraper\\logs\\'


In [2]:
import os
import glob
import time
import math

import unicodedata
from datetime import date
import re
import pandas as pd
from IPython.display import display
from fuzzywuzzy import fuzz
from ytmusicapi import YTMusic

## YTMusic API and Functions

In [3]:
def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
ytm = YTMusic(os.path.join(ytmusic_header_path, 'headers_auth.json'))
yt_res_cache = {}
yt_unmatched_cache = {}

## String Helper Functions

In [4]:
def scrub_title(title):
    title = title.lower().strip()
    title = unicodedata.normalize('NFKD', title).encode(
        'ascii', 'ignore').decode()
    title = title.replace('ft.', 'feat.')
    is_album = False
    for k in ['full album', '(album)', 'album stream']:
        if k in title:
            is_album = True
    for k in [
        '(official', '[official', 'official video', '[official', '[unoffical', '[free', '(free',
        '(explicit', '[video', '(video', '(music', '(nsfw', '(unoffical', '(live', '[live',
        '(video', 'music video', 'live video', '(original', '(lyric', 'lyric video', 
        '[full', '(full', '(album)', 'album stream', 'album review', '[thissongissick',
        '(prod', 'prod.', 'produced by', '[leak', '(from', '(lofi hip', '(remaster']:
        if k in title:
            title = title.split(k)[0]
    for k in ['[hd]', '[hq]', 'hd', 'hq']:
        if k in title:
            title=title.replace(k, '')
    return title, is_album

def strip_non_alphanumeric(value):
    value = str(re.sub('[^\\w\\s-]', ' ', str(value)))
    value = str(re.sub('[-\\s]+', ' ', str(value)))
    return value.strip()    

def extract_match_scores(query, match):
    q = strip_non_alphanumeric(query).lower()
    m = strip_non_alphanumeric(match).lower()
    return {
        'token_set_ratio': fuzz.token_set_ratio(q, m),
        # 'ratio': fuzz.ratio(q, m),
        'token_sort_ratio': fuzz.token_sort_ratio(q, m),
    }

# TODO FIRST
* finish reviewwing scraped tracks matches (delete good matches, then upload playlists, maybe merge uploading with next step)
* review 2021 refresh matches and uploadt to playlists
* run new 2021 subreddits, then review matches and upload playlists

## Parse Reddit .tsv and query YTMusic for Match

* Some reddit entries are albums, most are tracks
* lots of title 'scrubbing' before query to clean
* saves match and unmatched seperatly, caches query responses 
    * cache in above ytmusic api cell
* scores match using fuzz metrics
    * tries to automate passing macthes with score threshold


#### Last run 10/17/2021


In [61]:
# tmp

graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_scored_matches_and_graded_2021-10-16.tsv')
unmatched_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_failed_matches_2021-10-16.tsv')
historical_matches = pd.concat([
    pd.read_csv(graded_tsv, sep='\t', index_col=0), 
    pd.read_csv(unmatched_tsv, sep='\t', index_col=0)
    ])
historical_matches['match_date'] = '10-16-2021'
historical_matches['reddit_sub_yt_id'] = historical_matches['reddit_sub'] + ' | ' + historical_matches['ytmusic_key']

print(f'Loaded {len(historical_matches)} previous ytmusic matches')
url_history = frozenset(historical_matches.reddit_source_url)
print(f'{len(url_history)} unique urls in history')

graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_2021-refresh_ytmusic_scored_matches_2021-10-24.tsv')
unmatched_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_2021-refresh_ytmusic_failed_matches_2021-10-24.tsv')
recent_matches = pd.concat([
    pd.read_csv(graded_tsv, sep='\t', index_col=0), 
    pd.read_csv(unmatched_tsv, sep='\t', index_col=0)
    ])
recent_matches['match_date'] = '10-24-2021'
recent_matches['reddit_sub_yt_id'] = recent_matches['reddit_sub'] + ' | ' + recent_matches['ytmusic_key']

print(f'Loaded {len(recent_matches)} recent ytmusic matches')

# all_matches = pd.concat([
#     historical_matches, 
#     recent_matches
#     ]).sort_values('reddit_sub')
# print(f'Merged {len(all_matches)} all ytmusic matches')
# len(recent_matches.loc[recent_matches.reddit_sub_url_id.isin(historical_matches.reddit_sub_url_id)])
# all_matches['reddit_sub_url_id'] = all_matches['reddit_sub'] + '//' + all_matches['reddit_source_url']
# all_matches = all_matches.sort_values('match_date')
# print(len(all_matches))
# all_matches = all_matches.drop_duplicates(subset='reddit_sub_url_id', keep='first')
# print(len(all_matches))
# # all_matches  = all_matches.drop('reddit_sub_url_id', axis=1)
# recent_matches = all_matches.loc[all_matches.match_date == '10-24-2021']
# len(recent_matches)


Loaded 15165 previous ytmusic matches
13058 unique urls in history
Loaded 6448 recent ytmusic matches


In [64]:
len(recent_matches.loc[~recent_matches.reddit_sub_yt_id.isin(historical_matches.reddit_sub_yt_id)])


5722

In [15]:
reddit_subfolder = '2021_refresh'
tsv_path = os.path.join(reddit_log_path, reddit_subfolder)
log_every_n_matches = 1000
tail_n_entries = 10
fname_splitter = '_'
expected_fname_toks = 4
expected_cols = ['reddit', 'youtube_id', 'title', 'url', 'author', 'timestamp',
                 'description', 'likes', 'dislikes', 'num_comments', 'num_plays',
                 'is_media', 'thumb_url', 'num_views']

matched_entries = []
unmatched_entries = []
log_tsvs = sorted(glob.glob(os.path.join(tsv_path, '*.tsv')))
print(f'Found {len(log_tsvs)} reddit tsvs')
for i, tsv_file in enumerate(log_tsvs):
    name = os.path.splitext(os.path.basename(tsv_file))[0]
    toks = name.split(fname_splitter)
    if len(toks) != expected_fname_toks:
        print(
            f'skipping tsv without {expected_fname_toks} "{fname_splitter}" split toks: {tsv_file}')
        continue
    sub, agg, count, timestamp = toks
    df = pd.read_csv(tsv_file, sep='\t', index_col=0)
    if len(df) == 0:
        print(f'\n\nSkipping empty tsv: {tsv_file}')
        continue
    print(f'\n\n({i}/{len(log_tsvs)})  Loaded {len(df)} entries with {len(df.columns)} columns from {name}')
    assert set(df.columns) == set(expected_cols)

    # Loop thru entries
    for entry in df.itertuples():
        if 'youtube' not in entry.url and 'soundcloud' not in entry.url:
            print(f' skipping, Unknown url source {entry.url}')
        title_key, is_album = scrub_title(entry.title)

        # Check cache for saved YTMusic query response or previous match failures
        if entry.url in yt_res_cache:
            match = yt_res_cache[entry.url]
        elif entry.url in url_history:
            match = historical_matches.loc[historical_matches.reddit_source_url == entry.url]
            match = match[[c for c in match.columns if 'reddit_' not in c]]
            match = match.iloc[0].to_dict()
            yt_res_cache[entry.url] = match
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            unmatched_entries.append(yt_unmatched_cache[title_key])
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                if is_album:
                    res = ytm.search(query=title_key, filter='albums', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = True
                        match['ytmusic_title'] = ''
                        match['ytmusic_album'] = res.get('title', '')
                        match['ytmusic_albumId'] = res.get('browseId', '')
                        if 'artists' in res:
                            match['ytmusic_artists'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_album', '')}".lower()
                        match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_albumId','')}"

                else:
                    res = ytm.search(query=title_key, filter='songs', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = False
                        match['ytmusic_album'] = ''
                        if 'album' in res:
                            match['ytmusic_album'] = res['album'].get('name', '')
                            match['ytmusic_albumId'] = res['album'].get('id', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_title'] = res.get('title', '')
                        match['ytmusic_videoId'] = res.get('videoId', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')}".lower()
                        match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_videoId','')}"

            except Exception as e:
                print(f'Error with {entry.title}: {e}')
                pass

            # Store unmatched ytmusic query
            if len(res) == 0:
                unmatch = {}
                unmatch['reddit_title'] = entry.title
                unmatch['reddit_source_url'] = entry.url
                unmatch['reddit_key'] = title_key
                unmatch['reddit_sub'] = sub
                unmatch['reddit_sub_id'] = f'{sub}-{title_key}'
                unmatch['reddit_aggregator'] = agg
                unmatched_entries.append(unmatch)
                yt_unmatched_cache[title_key] = unmatch
                print(f'Skipping : {entry.title}')
                continue


            # Other ytmusic fields
            match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
            match['ytmusic_duration'] = res['duration']
            match['ytmusic_year'] = res['year']
            match['ytmusic_resultType'] = res['resultType']
            yt_res_cache[entry.url] = match

        # Reddit fields
        match['reddit_title'] = entry.title
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = entry.url
        match['youtube_videoId'] = entry.youtube_id

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Thresholds based on first round of manual quality check
            # see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
            # see: reddit_scraper\logs\ytmusic\*.png
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 70:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 90:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {entry.title}.\n  Match Error: {e}')

        # Update match results
        matched_entries.append(match)

        # if len(matched_entries) % log_every_n_matches == 0:
        #     cols = ['reddit_sub','reddit_aggregator','reddit_key', 'ytmusic_key', 'ytmusic_resultType']
        #     display(pd.DataFrame(matched_entries)[cols].tail(tail_n_entries))

# Process Matched entries
match_df = pd.DataFrame(matched_entries)
orig_len = len(match_df)
match_df = match_df.drop_duplicates(subset='ytmusic_playlist_id', keep='first')
print(f'Dropped {orig_len - len(match_df)} subreddit match duplicates')
print(f'\n\nSaving output tsvs to {reddit_log_path}')
match_file = os.path.join(reddit_log_path, 'ytmusic',
                          f'reddit_2021-refresh_ytmusic_scored_matches_{date.today()}.tsv')
match_df.to_csv(match_file, sep='\t', header=True)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
orig_len = len(unmatch_df)
unmatch_df = unmatch_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(unmatch_df)} subreddit unmatch duplicates')
unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_2021-refresh_ytmusic_failed_matches_{date.today()}.tsv')
unmatch_df.to_csv(unmatch_file, sep='\t', header=True)


Found 50 reddit tsvs


(0/50)  Loaded 268 entries with 14 columns from 90shiphop_top-all_1000_1635045534
Skipping : MF DOOM: In Tribute // 01-01-20


(1/50)  Loaded 191 entries with 14 columns from 90shiphop_top-year_1000_1635046186
Skipping: previously unmatched entry: mf doom: in tribute // 01-01-20
Skipping : Big L - Devil's Son Lyrics (Ft. Showbiz, Rosenberg and DJ Premier Reaction)


(2/50)  Loaded 94 entries with 14 columns from blues_top-all_1000_1635045461
Skipping : Christone “Kingfish” Ingram, 22nd Birthday Celebration (Concert Performance) Saturday, 1/16 @ 10p ET


(3/50)  Loaded 98 entries with 14 columns from blues_top-year_1000_1635046114
Skipping: previously unmatched entry: christone kingfish ingram, 22nd birtay celebration (concert performance) saturday, 1/16 @ 10p et
Skipping : Long Tall Woman (01-25/26-52)
Skipping : Swamp Family TV: Tedeschi Trucks Band 2/18/21 'The Fireside Sessions' LIVE Preview


(4/50)  Loaded 597 entries with 14 columns from chillmusic_top-all_

## Next do new subreddits

In [16]:
reddit_subfolder = '2021_new'
tsv_path = os.path.join(reddit_log_path, reddit_subfolder)
log_every_n_matches = 1000
tail_n_entries = 10
fname_splitter = '_'
expected_fname_toks = 4
expected_cols = ['reddit', 'youtube_id', 'title', 'url', 'author', 'timestamp',
                 'description', 'likes', 'dislikes', 'num_comments', 'num_plays',
                 'is_media', 'thumb_url', 'num_views']

matched_entries = []
unmatched_entries = []
log_tsvs = sorted(glob.glob(os.path.join(tsv_path, '*.tsv')))
print(f'Found {len(log_tsvs)} reddit tsvs')
for i, tsv_file in enumerate(log_tsvs):
    name = os.path.splitext(os.path.basename(tsv_file))[0]
    toks = name.split(fname_splitter)
    if len(toks) != expected_fname_toks:
        print(
            f'skipping tsv without {expected_fname_toks} "{fname_splitter}" split toks: {tsv_file}')
        continue
    sub, agg, count, timestamp = toks
    df = pd.read_csv(tsv_file, sep='\t', index_col=0)
    if len(df) == 0:
        print(f'\n\nSkipping empty tsv: {tsv_file}')
        continue
    print(f'\n\n({i}/{len(log_tsvs)})  Loaded {len(df)} entries with {len(df.columns)} columns from {name}')
    assert set(df.columns) == set(expected_cols)

    # Loop thru entries
    for entry in df.itertuples():
        if 'youtube' not in entry.url and 'soundcloud' not in entry.url:
            print(f' skipping, Unknown url source {entry.url}')
        title_key, is_album = scrub_title(entry.title)

        # Check cache for saved YTMusic query response or previous match failures
        if entry.url in yt_res_cache:
            match = yt_res_cache[entry.url]
        elif entry.url in url_history:
            match = historical_matches.loc[historical_matches.reddit_source_url == entry.url]
            match = match[[c for c in match.columns if 'reddit_' not in c]]
            match = match.iloc[0].to_dict()
            yt_res_cache[entry.url] = match
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            unmatched_entries.append(yt_unmatched_cache[title_key])
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                if is_album:
                    res = ytm.search(query=title_key, filter='albums', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = True
                        match['ytmusic_title'] = ''
                        match['ytmusic_album'] = res.get('title', '')
                        match['ytmusic_albumId'] = res.get('browseId', '')
                        if 'artists' in res:
                            match['ytmusic_artists'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_album', '')}".lower()
                        match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_albumId','')}"

                else:
                    res = ytm.search(query=title_key, filter='songs', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = False
                        match['ytmusic_album'] = ''
                        if 'album' in res:
                            match['ytmusic_album'] = res['album'].get('name', '')
                            match['ytmusic_albumId'] = res['album'].get('id', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_title'] = res.get('title', '')
                        match['ytmusic_videoId'] = res.get('videoId', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')}".lower()
                        match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_videoId','')}"

            except Exception as e:
                print(f'Error with {entry.title}: {e}')
                pass

            # Store unmatched ytmusic query
            if len(res) == 0:
                unmatch = {}
                unmatch['reddit_title'] = entry.title
                unmatch['reddit_source_url'] = entry.url
                unmatch['reddit_key'] = title_key
                unmatch['reddit_sub'] = sub
                unmatch['reddit_sub_id'] = f'{sub}-{title_key}'
                unmatch['reddit_aggregator'] = agg
                unmatched_entries.append(unmatch)
                yt_unmatched_cache[title_key] = unmatch
                print(f'Skipping : {entry.title}')
                continue


            # Other ytmusic fields
            match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
            match['ytmusic_duration'] = res['duration']
            match['ytmusic_year'] = res['year']
            match['ytmusic_resultType'] = res['resultType']
            yt_res_cache[entry.url] = match

        # Reddit fields
        match['reddit_title'] = entry.title
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = entry.url
        match['youtube_videoId'] = entry.youtube_id

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Thresholds based on first round of manual quality check
            # see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
            # see: reddit_scraper\logs\ytmusic\*.png
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 70:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 90:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {entry.title}.\n  Match Error: {e}')

        # Update match results
        matched_entries.append(match)

        # if len(matched_entries) % log_every_n_matches == 0:
        #     cols = ['reddit_sub','reddit_aggregator','reddit_key', 'ytmusic_key', 'ytmusic_resultType']
        #     display(pd.DataFrame(matched_entries)[cols].tail(tail_n_entries))

# Process Matched entries
match_df = pd.DataFrame(matched_entries)
orig_len = len(match_df)
match_df = match_df.drop_duplicates(subset='ytmusic_playlist_id', keep='first')
print(f'Dropped {orig_len - len(match_df)} subreddit match duplicates')
print(f'\n\nSaving output tsvs to {reddit_log_path}')
match_file = os.path.join(reddit_log_path, 'ytmusic',
                          f'reddit_2021-new_ytmusic_scored_matches_{date.today()}.tsv')
match_df.to_csv(match_file, sep='\t', header=True)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
orig_len = len(unmatch_df)
unmatch_df = unmatch_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(unmatch_df)} subreddit unmatch duplicates')
unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_2021-new_ytmusic_failed_matches_{date.today()}.tsv')
unmatch_df.to_csv(unmatch_file, sep='\t', header=True)


Found 140 reddit tsvs


(0/140)  Loaded 587 entries with 14 columns from 2000smusic_top-all_1000_1635050345


(1/140)  Loaded 541 entries with 14 columns from 2000smusic_top-year_1000_1635052628
Skipping : 50 Cent - P.I.M.P. (Ziggy Remix) [CBR XCLSV Video]
Skipping : Melvins - Electroretard (2001) [FULL ALBUM]
Skipping : Eddie Vedder w/ Tom Petty & The Heartbreakers - The Waiting - 7.03.06 -1080.HD


(2/140)  Loaded 540 entries with 14 columns from 2010smusic_top-all_1000_1635050333
Skipping : cheap thrills - sia - drum overdub cover - fantasia style improvisation
Skipping : Madeon - Pay No Mind (ft  Passion Pit) (BBC1 Radio Rip 2/9/2015)


(3/140)  Loaded 480 entries with 14 columns from 2010smusic_top-year_1000_1635052616
Skipping: previously unmatched entry: cheap thrills - sia - drum overdub cover - fantasia style improvisation
Skipping : Ed Sheeran - Shape Of You [UT99 - Unreal Tournament]
Skipping : summer - calvin harris - drum overdub cover - fantasia style improvisation
Error 

## Create YTMusic Playlists from matches

* manually marked each match as 'ok' (pass) or 'x' (fail)
    * see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
* computed match score using a few fuzz algos
    * set basic threshold to try to automate grading

The cells below:
1. Load the graded matches and split into pass / fail matches
2. Merges the failing matches with the unmatched reddit posts, then saves a new tsv with all unmatched 
3. Save the passing tracks to respective subreddit track playlists (split into like / unrated)
4. Save the passing albums to respective subreddit album playlists (split out liked tracks to merge with track playlist)

### Done
* use fuzzy id scoring to determine if match is passing
    * if failing add to unmatched entries
    * if passing add to ytmusic {r.subreddit} playlist
        * have one playlist for albums and another for tracks
* split out 'likes' for each {r.subreddit} ytmusic playlist
* handle passing tacks and albums by adding to subreddit playlist
* handle failed matches, att them to the unmaatched tsv with their scores and extra cols intact
    * maybe retry search? look and see if search query can be done better
    * if source is youtube add youtube link to correct ytmusic sub playlist (last resort)
        * add to albums vs track playist based on duration? 
#### Last run 10/17/2021

## Re-match (round 1)
reddit_scraper\logs\2021_new\PunkRock_top-all_1000_1635049772.tsv
reddit_scraper\logs\2021_new\PunkRock_top-year_1000_1635052086.tsv
reddit_scraper\logs\2021_new\witchHouse_top-year_1000_1635051978.tsv
reddit_scraper\logs\2021_new\witchHouse_top-all_1000_1635049613.tsv

In [ ]:
graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_scored_matches_and_graded_2021-10-16.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t', index_col=0)

# Drop duplicate entries (not needed in future tsvs, added dupe deleting in above cell)
graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId

# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
failing = graded_matches.loc[graded_matches.manual_check == 'x']

# Merge failing with unmatched and save new tsv
unmatched_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_failed_matches_2021-10-16.tsv')
unmatched_df = pd.read_csv(unmatched_tsv, sep='\t', index_col=0)
all_unmatched = pd.concat([failing, unmatched_df]).sort_values('reddit_sub')
print(f'Saving {len(all_unmatched)} unmatched reddit entries')
all_unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_ytmusic_failed_matches_graded_{date.today()}.tsv')
all_unmatched.to_csv(all_unmatch_file, sep='\t', header=True)

## Re-match (round 2)

graded 2021 refresh and more 2019-scraped

In [30]:
graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_scraped-2019_and_2021-refresh_ytmusic_scored_and_graded_matches_2021-10-24.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t')
graded_matches = graded_matches.sort_values('idx b')

# Drop duplicate entries (not needed in future tsvs, added dupe deleting in above cell)
graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId
print(len(graded_matches))
graded_matches = graded_matches.drop_duplicates('ytmusic_playlist_id')
print(len(graded_matches))

graded_matches.loc[graded_matches.is_album==True]

3618
3559


,idx b,idx a,manual_check,ytmusic_key,reddit_title,match_score_token_set_ratio,match_score_token_sort_ratio,match_quality,is_album,ytmusic_album,...,ytmusic_duration,ytmusic_year,ytmusic_resultType,reddit_key,reddit_sub,reddit_aggregator,reddit_file,reddit_source_url,youtube_id,Unnamed: 26
50,24.0,NaN,ok,- illmatic,Illmatic (Full Album),100.0,100.0,1.0,True,Illmatic,...,NaN,1994.0,album,illmatic,90shiphop,top-all,NaN,https://www.youtube.com/watch?v=oqVD2yQqH7M&fe...,oqVD2yQqH7M,Nas
3480,1304.0,NaN,ok,- meridian,Rohne - Meridian [Full Album],100.0,73.0,1.0,True,Meridian,...,NaN,2018.0,album,rohne - meridian,triphop,top-all,NaN,https://www.youtube.com/watch?v=FozmE-RbpC0,FozmE-RbpC0,Rohne
3445,3531.0,NaN,x,- ???????,???? - ???????/Birth of a New Day (Full Album)...,0.0,0.0,0.0,True,???????,...,NaN,2015.0,album,2814 - /birth of a new day,futurebeats,top-year,NaN,https://www.youtube.com/watch?v=F9L4q-0Pi4E,F9L4q-0Pi4E,2814
1857,3785.0,NaN,ok,"- all those estranged things i have created, l...",Grand Inc - All Those Estranged Things I Have ...,100.0,91.0,1.0,True,"All Those Estranged Things I Have Created, Loo...",...,NaN,2021.0,album,grand inc - all those estranged things i have ...,futurebeats,top-year,NaN,https://www.youtube.com/watch?v=-zZToo9_OeE,-zZToo9_OeE,Grand Inc
3202,4341.0,NaN,x,- sunday avenue,DJ Hollow - Nightlife [Lofi House / Garage] (F...,24.0,24.0,0.0,True,Sunday Avenue,...,NaN,2017.0,album,dj hollow - nightlife [lofi house / garage],futuregarage,top-year,NaN,https://www.youtube.com/watch?v=GN8A8lNjnfE&ab...,GN8A8lNjnfE,DJ Boring
3490,5134.0,NaN,ok,- o.s.t.,People Under The Stairs - O.S.T. [Full Album],100.0,29.0,0.0,True,O.S.T.,...,NaN,2002.0,album,people under the stairs - o.s.t.,hiphop,top-year,NaN,https://www.youtube.com/watch?v=RwKlQ9k7Lzk&fe...,RwKlQ9k7Lzk,People Under The Stairs
3444,6512.0,NaN,x,- ?????,MF DOOM X Tatsuro Yamashita (Full Album),0.0,0.0,0.0,True,?????,...,NaN,2016.0,album,mf doom x tatsuro yamashita,jazz,top-year,NaN,https://www.youtube.com/watch?v=bqkOQ46lxj8,bqkOQ46lxj8,???
3010,6532.0,NaN,x,- kofi,"Donald Byrd ?""Kofi"" Full Album",100.0,26.0,0.0,True,Kofi,...,NaN,1971.0,album,"donald byrd ""kofi"" full album",jazz,top-year,NaN,https://www.youtube.com/watch?v=0Ve415qxUbs,0Ve415qxUbs,Donald Byrd
3483,6573.0,NaN,ok,- red clay,FREDDIE HUBBARD - Red Clay LP 1970 Full Album,100.0,31.0,0.0,True,Red Clay,...,NaN,1987.0,album,freddie hubbard - red clay lp 1970 full album,jazz,top-year,NaN,https://www.youtube.com/watch?v=KLc8vFT8y7Q,KLc8vFT8y7Q,Freddie Hubbard
3051,6670.0,NaN,x,- promises,"Floating Points, Pharoah Sanders & The London ...",100.0,21.0,0.0,True,Promises,...,NaN,2021.0,album,"floating points, pharoah sanders & the london ...",jazznoir,top-year,NaN,https://www.youtube.com/watch?v=Mn8x0QbN4f8,Mn8x0QbN4f8,"Floating Points, Pharoah Sanders & The London ..."


In [31]:
graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_scraped-2019_and_2021-refresh_ytmusic_scored_and_graded_matches_2021-10-24.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t')
graded_matches = graded_matches.sort_values('idx b')

# Drop duplicate entries (not needed in future tsvs, added dupe deleting in above cell)
graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId
print(len(graded_matches))
graded_matches = graded_matches.drop_duplicates('ytmusic_playlist_id')
print(len(graded_matches))

# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
failing = graded_matches.loc[graded_matches.manual_check == 'x']


# Merge failing with unmatched and save new tsv
# Prevsioulsy merged unmatch + failed from 10-16
unmatched_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_failed_matches_graded_2021-10-16.tsv')
unmatched_df = pd.read_csv(unmatched_tsv, sep='\t', index_col=0)
print('failed to match 10-17: ',len(unmatched_df))

# Failed to match 10-24
failed_match_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_2021-refresh_ytmusic_failed_matches_2021-10-24.tsv')
failed_df = pd.read_csv(failed_match_tsv, sep='\t', index_col=0)
print('failed to match 10-24: ',len(failed_df))

# Failed after grading 10-24
failing_recent = failing.loc[~failing.reddit_file.isna()]
print('failed from grading 10-24: ',len(failing_recent))

# MERGE
all_unmatched = pd.concat([unmatched_df, failing_recent, failed_df]).sort_values('reddit_sub')
print(f'Saving {len(all_unmatched)} unmatched reddit entries')
all_unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_ytmusic_failed_matches_graded_{date.today()}.tsv')
all_unmatched.to_csv(all_unmatch_file, sep='\t', header=True)


3618
3559
failed to match 10-17:  4911
failed to match 10-24:  505
failed from grading 10-24:  316
Saving 5732 unmatched reddit entries


## Subreddit playlists for passing tracks

(same code for round 1 and 2, just clear completed [])

In [33]:
N=5
LIMIT = 900
passing_tracks = passing.loc[passing.is_album == False]
completed = []
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'futuregarage', 'hiphop', 'hiphopheads', 'indie', 'indieheads', 'indierock', 'jazz', 'jazznoir', 'jazzyhiphop', 'lofihiphop']
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'futuregarage', 'hiphop', 'hiphop101', 'hiphopheads', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'nudisco', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic']
# Note: r/song doesnt work probably too big? > 1000 tracks? maybe not
for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing tracks')
    vids = df.ytmusic_videoId.unique().tolist()
    title=f'x_r.{sub}_tracks_v3'
    desc = f'Matched {len(vids)} tracks from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} tracks playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    if len(tracks) < 10:
        print(f'Not enough tracks to split into like dislike: count = {len(tracks)}')
        continue

    # Create Like subset
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)
    
    # Create Unrated Radio subset
    unrated_tracks = tracks.loc[tracks['likeStatus'] != 'LIKE']
    vids = unrated_tracks.videoId.unique().tolist()
    desc = f'Unrated radio subset of {len(vids)} entries from: {desc}'
    radio_pl_id = ytm.create_playlist(title=f'{title}_radio', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} INDIFFERENT r.{sub} radio tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Delete original playlist now thet like/unrated is split
    ytm.delete_playlist(pl_id)
    print(f'Deleted r/{sub} tracks playlist with id: {pl_id}')
    completed.append(sub)
    



Generating r.90shiphop ytmusic playlist for 150 passing tracks
Saved 150 r.90shiphop tracks playlist with id: PLWptjpDqazOy6LK2znQ_YaI7cnIHwoiOY, waiting 5 seconds...
Filtered 45 LIKE r.90shiphop tracks playlist with id: PLWptjpDqazOz8jGQ8Gx37fugnnC4t1DWH, waiting 5 seconds...
Filtered 105 INDIFFERENT r.90shiphop radio tracks playlist with id: PLWptjpDqazOz8jGQ8Gx37fugnnC4t1DWH, waiting 5 seconds...
Deleted r/90shiphop tracks playlist with id: PLWptjpDqazOy6LK2znQ_YaI7cnIHwoiOY

Generating r.blues ytmusic playlist for 60 passing tracks
Saved 60 r.blues tracks playlist with id: PLWptjpDqazOxZUvs7aNCLL5mQ65fLo4SS, waiting 5 seconds...
Filtered 4 LIKE r.blues tracks playlist with id: PLWptjpDqazOymZfL-7X9OIzPUqtmoi5d6, waiting 5 seconds...
Filtered 56 INDIFFERENT r.blues radio tracks playlist with id: PLWptjpDqazOymZfL-7X9OIzPUqtmoi5d6, waiting 5 seconds...
Deleted r/blues tracks playlist with id: PLWptjpDqazOxZUvs7aNCLL5mQ65fLo4SS

Generating r.chillmusic ytmusic playlist for 23 passing

## Subreddit playlists for passing albums
(same code for round 1 and 2, just clear completed [])

In [34]:
SLEEP_TIME=5
LIMIT = 900
passing_tracks = passing.loc[passing.is_album == True]
completed= []
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'hiphop', 'hiphop101', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic', 'triphop']

for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing albums')
    vids = set()
    for row in df.itertuples():
        print(f' Adding album: {row.ytmusic_album}')
        for track in  ytm.get_album(row.ytmusic_albumId).get('tracks', []):
            vids.add(track['videoId'])
    vids = list(vids)

    title=f'x_r.{sub}_albums_v3'
    desc = f'Matched {len(vids)} albums from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} albums playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(SLEEP_TIME)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    # Create Like subset to merge with tracks playlist
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_tracks_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(SLEEP_TIME)
    completed.append(sub)




Generating r.90shiphop ytmusic playlist for 1 passing albums
 Adding album: Illmatic
Saved 10 r.90shiphop albums playlist with id: PLWptjpDqazOxW7XnUh3a_eYz5dlBWnYjD, waiting 5 seconds...
Filtered 6 LIKE r.90shiphop tracks playlist with id: PLWptjpDqazOz1BqADe2-50Mok7fH0TIz-, waiting 5 seconds...

Generating r.futurebeats ytmusic playlist for 1 passing albums
 Adding album: All Those Estranged Things I Have Created, Look At Me
Saved 10 r.futurebeats albums playlist with id: PLWptjpDqazOztvP-UTgoGV1RPK_aJ_mU5, waiting 5 seconds...
Filtered 0 LIKE r.futurebeats tracks playlist with id: PLWptjpDqazOyBz5HinhE1ZM5yxSStan5E, waiting 5 seconds...

Generating r.hiphop ytmusic playlist for 1 passing albums
 Adding album: O.S.T.
Saved 20 r.hiphop albums playlist with id: PLWptjpDqazOzswZK5yy8ohrrnhg0BnmXI, waiting 5 seconds...
Filtered 2 LIKE r.hiphop tracks playlist with id: PLWptjpDqazOwAxPjNYus0yDftBNTp_SVX, waiting 5 seconds...

Generating r.jazz ytmusic playlist for 1 passing albums
 Addin